# ROC-AUC: Золотой стандарт классификации

Алгоритмы машинного обучения (например, логистическая регрессия или градиентный бустинг) редко выдают жесткий ответ «Да» или «Нет». Обычно они выдают **вероятность** — например, «этот клиент вернет кредит с вероятностью 0.85».

Чтобы перевести вероятность в финальное решение (выдать кредит или отказать), нам нужен **порог отсечения (threshold)**. По умолчанию он равен 0.5. Но если мы изменим порог на 0.8, метрики _Precision_ и _Recall_ кардинально изменятся.

**Главная проблема:** Как оценить качество самой модели, не привязываясь к конкретному порогу? Именно эту задачу решает ROC-AUC.

## 1. Как читать график (ROC-кривая)

**ROC-кривая (Receiver Operating Characteristic)** — это график, который показывает, как меняется качество модели при переборе всех возможных порогов отсечения (от 0.0 до 1.0).

У графика есть две оси:

- **Ось Y (TPR - True Positive Rate):** Это тот же самый _Recall_ (Полнота). Доля найденных реальных «единиц». Мы хотим, чтобы она стремилась к $100\%$ ($1.0$).
    
    $$TPR = \frac{TP}{TP + FN}$$
- **Ось X (FPR - False Positive Rate):** Доля ложных тревог. Процент объектов класса «ноль», которые модель ошибочно назвала «единицей». Мы хотим, чтобы она стремилась к $0\%$ ($0.0$).
    
    $$FPR = \frac{FP}{FP + TN}$$

**Как строится кривая:**

1. Мы ставим порог $1.0$ (никто не получает класс «1»). Модель не делает ложных тревог ($FPR = 0$), но и никого не находит ($TPR = 0$). Это точка $(0, 0)$ в левом нижнем углу.
    
2. Мы постепенно снижаем порог. Начинают появляться как верные попадания, так и ложные тревоги. Линия ползет вверх и вправо.
    
3. При пороге $0.0$ всем объектам присваивается класс «1». Модель нашла всех ($TPR = 1$), но подняла 100% ложных тревог ($FPR = 1$). Это точка $(1, 1)$ в правом верхнем углу.
    

**Диагональ графика** (из левого нижнего в правый верхний угол) — это случайное угадывание (бросок монетки). Хорошая модель всегда должна выгибаться дугой **выше** этой диагонали, стремясь в идеальный левый верхний угол (где $TPR=1$, а $FPR=0$).

## 2. Что такое AUC?

Сама по себе кривая — это просто картинка. Чтобы сравнивать модели программно, нужно одно число.

**AUC (Area Under Curve)** — это площадь под ROC-кривой.

- $AUC = 0.5$ (Площадь под диагональю) — Модель абсолютно бесполезна, она гадает наугад.
    
- $AUC = 1.0$ (Вся площадь квадрата) — Идеальная модель, которая безошибочно разделяет классы.
    
- $AUC \approx 0.7 - 0.8$ — Нормальная, рабочая модель.
    
- $AUC \approx 0.8 - 0.9$ — Отличная модель.
    
- $AUC < 0.5$ — Модель работает "наоборот" (ошибается чаще, чем гадает). Если поменять её ответы на противоположные, она станет хорошей.
    

## 3. Магическая интерпретация AUC (Почему это золотой стандарт?)

У площади под кривой (AUC) есть гениальное математическое свойство, которое делает её лучшей метрикой для бизнеса:

> **AUC — это вероятность того, что если мы возьмем один случайный "хороший" объект и один случайный "плохой", модель даст "хорошему" объекту балл выше, чем "плохому".**

**Пример из жизни:** Представьте, вы разработали систему кредитного скоринга с $AUC = 0.85$. Вы берете из истории банка одного случайного дефолтника (класс 0) и одного случайного человека, вернувшего долг (класс 1). С вероятностью $85\%$ ваша модель присвоила честному заемщику скоринговый балл **выше**, чем дефолтнику.

### Почему бизнес любит ROC-AUC:

1. **Оценка ранжирования, а не угадывания.** Модели часто используют для сортировки (например, кому из клиентов позвонить первому, чтобы предложить услугу). ROC-AUC идеально показывает, насколько хорошо модель сортирует клиентов от наиболее вероятных к наименее вероятным.
    
2. **Игнорирование дисбаланса.** В отличие от _Accuracy_, ROC-AUC не ломается, если у вас 99% нулей и 1% единиц.
    
3. **Разделение труда.** Дата-саентист может обучить модель с высоким AUC (обеспечить хорошее разделение классов), а бизнес-заказчик потом сам выберет нужный порог отсечения (Threshold) на основе своих финансовых рисков, двигаясь вдоль ROC-кривой.